In [6]:
import pandas as pd
import numpy as np

#Filters out and keeps Pittsburgh Neighborhoods
df = pd.read_csv("Neighborhood-Profiles.csv")
df = df[df['GeographyType'] == 'neighborhood'].copy()

col_2012 = 'Var_2012_income_Per_2'
col_2022 = 'Var_2022_income_Per_2'

report = df[['NeighborhoodGroup', col_2012, col_2022]].dropna().copy()
report = report.set_index('NeighborhoodGroup')

with np.errstate(divide='ignore', invalid='ignore'):
    growth_12_22 = ((report[col_2022] - report[col_2012])/report[col_2012])*100

growth_12_22 = growth_12_22.replace([np.inf, -np.inf], np.nan).fillna(0)
report['2012-2022 Trend'] = growth_12_22

def format_trend(val):
    if pd.isna(val) or val == 0:
        return "0.0%"
    if(val>0):
        return f"+{val:.1f}%"
    else:
        return f"{val:.1f}%"
report = report.sort_values(by=col_2022, ascending=True)
report['2012-2022 Trend'] = report['2012-2022 Trend'].apply(format_trend)

pd.set_option('display.max_rows', None)
report

,Var_2012_income_Per_2,Var_2022_income_Per_2,2012-2022 Trend
NeighborhoodGroup,,,
South Shore,0.000000,0.000000,0.0%
Highland Park,0.246337,0.076230,-69.1%
Point Breeze North,0.367761,0.078081,-78.8%
Strip District,0.074468,0.080600,+8.2%
Swisshelm Park,0.169257,0.082077,-51.5%
Lower Lawrenceville,0.450388,0.119550,-73.5%
Upper Lawrenceville,0.383838,0.126538,-67.0%
Morningside,0.266753,0.132673,-50.3%
Banksville,0.151031,0.136895,-9.4%


In [9]:
pct_change = report['2012-2022 Trend'].astype(str).str.replace('%', '').str.replace('+', '').astype(float)
report['Divisor Baseline'] = 100 - pct_change

report['Affordability Score'] = report['Divisor Baseline']/report['Var_2022_income_Per_2']

report = report.sort_values(by='Affordability Score', ascending=False)
report['Rank'] = range(1, len(report) + 1)

report[['Rank', 'Var_2022_income_Per_2', '2012-2022 Trend', 'Affordability Score']]

,Rank,Var_2022_income_Per_2,2012-2022 Trend,Affordability Score
NeighborhoodGroup,,,,
South Shore,1,0.000000,0.0%,inf
Point Breeze North,2,0.078081,-78.8%,2289.932533
Highland Park,3,0.076230,-69.1%,2218.278063
Swisshelm Park,4,0.082077,-51.5%,1845.826529
Lower Lawrenceville,5,0.119550,-73.5%,1451.276467
Upper Lawrenceville,6,0.126538,-67.0%,1319.763885
Strip District,7,0.080600,+8.2%,1138.960459
Morningside,8,0.132673,-50.3%,1132.858212
Beechview,9,0.144417,-54.3%,1068.430250
